# Tutorial: Vision IP Demo

Audience:
- Students testing the 3x3 filter path before switching to the board.

Prerequisites:
- Python 3 with NumPy.
- Sample PGM images in `data/input_images/`.

Learning goals:
- Load a sample frame.
- Push it through the filter helper.
- Compare the hardware path against the software reference.


## Setup


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'software').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from software.pynq.overlay import AcceleratorOverlay
from software.pynq.benchmarks import benchmark_filter
from software.pynq.reference import load_pgm, process_frame_reference
from software.pynq.registers import KernelMode


## Load a test image and run the Sobel mode


In [ ]:
suite = AcceleratorOverlay(use_mock=True)
image_path = repo_root / 'data' / 'input_images' / 'checkerboard_64.pgm'
frame = load_pgm(image_path)
hw_edges = suite.vision.process_image_hw(frame, mode=KernelMode.SOBEL_MAG)
sw_edges = process_frame_reference(frame, KernelMode.SOBEL_MAG)
hw_edges.shape, int(hw_edges.max())


## Check that the mock hardware path matches the software reference


In [ ]:
bool((hw_edges == sw_edges).all())


## Save one output image


In [ ]:
out_path = repo_root / 'data' / 'output_images' / 'notebook_sobel_output.pgm'
suite.vision.save_gray_image(out_path, hw_edges)
out_path


## Benchmark the filter path


In [ ]:
benchmark_filter(frame, suite, repeat=8)
